# Classificacao Com rede Neural

## Coletando os dados

In [1]:
import numpy as np
import machinelearning_mnpkllps.Preprocessing as PP
import machinelearning_mnpkllps.Resampling as RA
import machinelearning_mnpkllps.NeuralNetwork as NN
import machinelearning_mnpkllps.Metrics as MT

np.set_printoptions(threshold=100)

In [2]:
# Leitura dos dados
dados = np.genfromtxt('../data/PARTICIPANTES_2025.csv', 
                      delimiter=';', 
                      skip_header=1, 
                      encoding='latin-1',
                      dtype=str,
                      usecols=(2, 3, 4, 5, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37))

# Colunas selecionadas:
# TP_FAIXA_ETARIA, TP_SEXO, TP_ESTADO_CIVIL, TP_COR_RACA, Q001, Q002, Q003, Q004, Q006, Q007, Q008, Q009, Q010, Q011, Q012, Q013, Q014, Q015, Q016, Q017, Q018, Q019, Q020, Q021, Q022, Q023

In [3]:
for i in range(25):
    y = dados[:, i]
    
    nan_y = np.sum((y == '') | (y == 'nan') | (y == 'None'))

    print(f"Valores NaN/vazios na coluna {i}: {nan_y}")

Valores NaN/vazios na coluna 0: 0
Valores NaN/vazios na coluna 1: 0
Valores NaN/vazios na coluna 2: 0
Valores NaN/vazios na coluna 3: 0
Valores NaN/vazios na coluna 4: 0
Valores NaN/vazios na coluna 5: 0
Valores NaN/vazios na coluna 6: 0
Valores NaN/vazios na coluna 7: 0
Valores NaN/vazios na coluna 8: 0
Valores NaN/vazios na coluna 9: 0
Valores NaN/vazios na coluna 10: 0
Valores NaN/vazios na coluna 11: 0
Valores NaN/vazios na coluna 12: 0
Valores NaN/vazios na coluna 13: 0
Valores NaN/vazios na coluna 14: 0
Valores NaN/vazios na coluna 15: 0
Valores NaN/vazios na coluna 16: 0
Valores NaN/vazios na coluna 17: 0
Valores NaN/vazios na coluna 18: 0
Valores NaN/vazios na coluna 19: 0
Valores NaN/vazios na coluna 20: 0
Valores NaN/vazios na coluna 21: 0
Valores NaN/vazios na coluna 22: 0
Valores NaN/vazios na coluna 23: 0
Valores NaN/vazios na coluna 24: 0


In [4]:
data = dados.copy()

# Identifica células vazias

linhas_validas = np.all(
    (data != '') & (np.char.lower(data.astype(str)) != 'nan'),
    axis=1
)

n = int(3e5)
clean_data = data[linhas_validas]

_, clean_data, _, _ = PP.train_test_split(clean_data, clean_data, test_size=0.1)


print(f"Número de linhas originais: {data.shape[0]}")
print(f"Número de linhas da base limpa: {clean_data.shape[0]}")

Número de linhas originais: 4810772
Número de linhas da base limpa: 481077


### Arrumando preditores categoricos

In [5]:
col = {
    'TP_FAIXA_ETARIA': 0,
    'TP_SEXO': 1,
    'TP_ESTADO_CIVIL': 2,
    'TP_COR_RACA': 3,
    'Q001': 4,
    'Q002': 5,
    'Q003': 6,
    'Q004': 7,
    'Q006': 8,
    'Q007': 9,
    'Q008': 10,
    'Q009': 11,
    'Q010': 12,
    'Q011': 13,
    'Q012': 14,
    'Q013': 15,
    'Q014': 16,
    'Q015': 17,
    'Q016': 18,
    'Q017': 19,
    'Q018': 20,
    'Q019': 21,
    'Q020': 22,
    'Q021': 23,
    'Q022': 24,
    'Q023': 25
    }

In [6]:
display(clean_data)
X = np.vectorize(
    lambda x: int(x) if x.isdigit() else ord(x) - ord('A') + 1
)(clean_data).copy()

col = {
    'TP_FAIXA_ETARIA': 0,
    'TP_SEXO': 1,
    'TP_ESTADO_CIVIL': 2,
    'TP_COR_RACA': 3,
    'Q001': 4,
    'Q002': 5,
    'Q003': 6,
    'Q004': 7,
    'Q006': 8,
    'Q007': 9,
    'Q008': 10,
    'Q009': 11,
    'Q010': 12,
    'Q011': 13,
    'Q012': 14,
    'Q013': 15,
    'Q014': 16,
    'Q015': 17,
    'Q016': 18,
    'Q017': 19,
    'Q018': 20,
    'Q019': 21,
    'Q020': 22,
    'Q021': 23,
    'Q022': 24,
    'Q023': 25
    }


array([['3', 'F', '1', ..., 'A', 'C', 'A'],
       ['2', 'F', '1', ..., 'A', 'D', 'A'],
       ['2', 'F', '1', ..., 'A', 'C', 'A'],
       ...,
       ['2', 'M', '1', ..., 'A', 'E', 'A'],
       ['4', 'F', '1', ..., 'A', 'D', 'A'],
       ['5', 'F', '1', ..., 'B', 'D', 'A']],
      shape=(481077, 26), dtype='<U2')

In [7]:
X_c = list(X.T.copy()).copy()
n = 12
X_c[1] = PP.CategoricalEncoder(drop_first=True).fit_transform(X_c[1])
X_c[2] = PP.CategoricalEncoder(drop_first=True).fit_transform(X_c[2])
X_c[3] = PP.CategoricalEncoder(drop_first=True).fit_transform(X_c[3])
X_c[4] -= 1
X_c[5] -= 1
X_c[6] -= 1
X_c[7] -= 1
X_c[8] -= 1
X_c[9] -= 1
X_c[10] -= 1
X_c[11] -= 1
X_c[12] -= 1
X_c[13] -= 1
X_c[14] -= 1
X_c[15] -= 1
X_c[16] -= 1
X_c[17] -= 1
X_c[18] -= 1
X_c[19] -= 1
X_c[20] -= 1
X_c[21] -= 1
X_c[22] -= 1
X_c[23] -= 1
X_c[24] -= 1
X_c[25] = PP.CategoricalEncoder(drop_first=True).fit_transform(X_c[25])


for i in range(len(X_c)):
    print(f"Coluna {i}")
    display(np.unique(X_c[i]))

Coluna 0


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20])

Coluna 1


array([0., 1.])

Coluna 2


array([0., 1.])

... (42 outputs left out)

Coluna 24


array([0, 1, 2, 3, 4])

Coluna 25


array([0., 1.])

In [8]:
X = np.concatenate([X_c[i].reshape(-1, 1) if X_c[i].ndim == 1 else X_c[i] for i in [1, 2, 3, 4, 5, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]], axis=1)
y = np.concatenate([X_c[i].reshape(-1, 1) if X_c[i].ndim == 1 else X_c[i] for i in [6, 7, 9]], axis=1)

display(X)
display(y)

array([[0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [1., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.]], shape=(481077, 33))

array([[5, 1, 2],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [0, 0, 1],
       [2, 2, 1],
       [1, 1, 3]], shape=(481077, 3))

In [9]:
for i in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
          17, 18, 19, 20, 21, 22, 23, 24, 25]:
    print(i, X_c[i].min(), X_c[i].max())

print("X:", X.min(), X.max())
print("y:", y.min(), y.max())

1 0.0 1.0
2 0.0 1.0
3 0.0 1.0
4 0 7
5 0 7
6 0 5
7 0 5
8 0 1
9 0 16
10 0 3
11 0 3
12 0 3
13 0 3
14 0 3
15 0 3
16 0 1
17 0 1
18 0 1
19 0 1
20 0 3
21 0 1
22 0 1
23 0 4
24 0 4
25 0.0 1.0
X: 0.0 7.0
y: 0 16


## Classificações

In [10]:
x_train, x_aux, y_train, y_aux = PP.train_test_split(X, y, test_size=0.40)
x_valid, x_test, y_valid, y_test = PP.train_test_split(x_aux, y_aux, test_size=0.50)

display(x_train.shape, y_train.shape)
display(x_valid.shape, y_valid.shape)
display(x_test.shape, y_test.shape)

(288646, 33)

(288646, 3)

(96215, 33)

(96215, 3)

(96216, 33)

(96216, 3)

### Classificação Classe de emprego do pai

In [11]:
y_q3_train = PP.CategoricalEncoder(drop_first=True).fit_transform(y_train[:, 0].reshape(-1))

display(y_q3_train)

nn_q3 = NN.NeuralNetwork([33, 64, 32, 5], 'sigmoid', 'sigmoid', bias_interval=(0, 0), weight_interval=(-0.1, 0.1))
nn_q3.fit(x_train, y_q3_train, learn_rate=0.01, batch=64, epochs=100, debug=True, tol=1e-8)


array([[0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       ...,
       [0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0.]], shape=(288646, 5))

Iniciando epoca 0, com erro 1.1941070763324033
Iniciando epoca 1, com erro 0.6666031729158489
Iniciando epoca 2, com erro 0.6663279729464517
Iniciando epoca 3, com erro 0.6660480957355368
Iniciando epoca 4, com erro 0.665690011964343
Iniciando epoca 5, com erro 0.6652706532941252
Iniciando epoca 6, com erro 0.6646116916156225
Iniciando epoca 7, com erro 0.6636776448217339
Iniciando epoca 8, com erro 0.6623393394555585
Iniciando epoca 9, com erro 0.6604337106406757
Iniciando epoca 10, com erro 0.6577902011282976
Iniciando epoca 11, com erro 0.6544303705938226
Iniciando epoca 12, com erro 0.6504706537189575
Iniciando epoca 13, com erro 0.6461942001159725
Iniciando epoca 14, com erro 0.6415778896142134
Iniciando epoca 15, com erro 0.6367660635420613
Iniciando epoca 16, com erro 0.6320089261394898
Iniciando epoca 17, com erro 0.627465472501124
Iniciando epoca 18, com erro 0.6231098261805949
Iniciando epoca 19, com erro 0.6187908514981665
Iniciando epoca 20, com erro 0.6149890824645815
Inic

### Classificação Classe de emprego do mae

In [12]:
y_q4_train = PP.CategoricalEncoder(drop_first=True).fit_transform(y_train[:, 1].reshape(-1))

display(y_q4_train)

nn_q4 = NN.NeuralNetwork([33, 64, 32, 5], 'sigmoid', 'sigmoid', bias_interval=(0, 0), weight_interval=(-0.1, 0.1))
nn_q4.fit(x_train, y_q4_train, learn_rate=0.01, batch=64, epochs=100, debug=True, tol=1e-8)

array([[0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       ...,
       [0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0.]], shape=(288646, 5))

Iniciando epoca 0, com erro 1.2267421347885432
Iniciando epoca 1, com erro 0.6215773126607481
Iniciando epoca 2, com erro 0.6210363271564339
Iniciando epoca 3, com erro 0.6201713692175813
Iniciando epoca 4, com erro 0.6186092692283225
Iniciando epoca 5, com erro 0.6155074094744735
Iniciando epoca 6, com erro 0.6095236368324718
Iniciando epoca 7, com erro 0.5999626387421599
Iniciando epoca 8, com erro 0.588260528114869
Iniciando epoca 9, com erro 0.5795383890669714
Iniciando epoca 10, com erro 0.5752854623519205
Iniciando epoca 11, com erro 0.5728147915273587
Iniciando epoca 12, com erro 0.5710704170752499
Iniciando epoca 13, com erro 0.569281580524353
Iniciando epoca 14, com erro 0.5678530264682372
Iniciando epoca 15, com erro 0.5666656436378553
Iniciando epoca 16, com erro 0.5654788401852807
Iniciando epoca 17, com erro 0.564506087750453
Iniciando epoca 18, com erro 0.5635858767575744
Iniciando epoca 19, com erro 0.5627511848457646
Iniciando epoca 20, com erro 0.561874772038765
Inicia

### Classificação Classe de Renda total da familia

In [15]:
y_q7_train = PP.CategoricalEncoder(drop_first=True).fit_transform(y_train[:, 2].reshape(-1))

display(y_q7_train)

nn_q7 = NN.NeuralNetwork([33, 64, 32, 16], 'sigmoid', 'sigmoid', bias_interval=(0, 0), weight_interval=(-0.1, 0.1))
nn_q7.fit(x_train, y_q7_train, learn_rate=0.01, batch=64, epochs=100, debug=True, tol=1e-8)

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(288646, 16))

Iniciando epoca 0, com erro 4.087220140781428
Iniciando epoca 1, com erro 0.733948462963264
Iniciando epoca 2, com erro 0.7330288547141901
Iniciando epoca 3, com erro 0.7322911587213388
Iniciando epoca 4, com erro 0.7313479712452797
Iniciando epoca 5, com erro 0.7290513935813845
Iniciando epoca 6, com erro 0.7249856161371726
Iniciando epoca 7, com erro 0.7170717836454412
Iniciando epoca 8, com erro 0.7044452751484938
Iniciando epoca 9, com erro 0.6918610763191484
Iniciando epoca 10, com erro 0.6849414221599337
Iniciando epoca 11, com erro 0.6816545365837182
Iniciando epoca 12, com erro 0.679849346422563
Iniciando epoca 13, com erro 0.6787204015214107
Iniciando epoca 14, com erro 0.6778554089461977
Iniciando epoca 15, com erro 0.6771509295722229
Iniciando epoca 16, com erro 0.6764619148285365
Iniciando epoca 17, com erro 0.6759678985220676
Iniciando epoca 18, com erro 0.6754257585871176
Iniciando epoca 19, com erro 0.67506797731923
Iniciando epoca 20, com erro 0.6745958227738573
Inician

## Criando classe de comparação

In [17]:
y_q3_pred = nn_q3.prevision(x_valid, classify=True)
y_q4_pred = nn_q4.prevision(x_valid, classify=True)
y_q7_pred = nn_q7.prevision(x_valid, classify=True)

y_q3 = y_valid[:, 0]
y_q4 = y_valid[:, 1]
y_q7 = y_valid[:, 2]

display(y_q3_pred, y_q3_pred, y_q3_pred)

ocupacao = np.isin(y_q3, [3, 4]) | np.isin(y_q4, [3, 4])
renda = y_q7 >= 6

classe = (ocupacao & renda).astype(int)

ocupacao_pred = np.isin(y_q3_pred, [3, 4]) | np.isin(y_q4_pred, [3, 4])
renda_pred = y_q7_pred >= 6

classe_pred = (ocupacao_pred & renda_pred).astype(int)

display(np.array(classe_pred), np.array(classe))

float(MT.acuracia(np.array(classe_pred), np.array(classe)))

array([2, 1, 2, ..., 1, 4, 0], shape=(96215,))

array([2, 1, 2, ..., 1, 4, 0], shape=(96215,))

array([2, 1, 2, ..., 1, 4, 0], shape=(96215,))

array([0, 0, 0, ..., 0, 0, 0], shape=(96215,))

array([0, 0, 1, ..., 0, 0, 0], shape=(96215,))

0.8374785636335291